# My First Model Compared with the Baseline

I am continuing the **Refresh / Content Opportunity Scoring** lane. I want to know whether a small learned model orders review candidates more usefully than the fixed Week 4 rule. The outcome is later impression decline, not whether editing a page would work.

The model, baseline, labels, feature window, eligible pages, and review budget stay aligned. This March comparison is development evidence: the month and split have already been inspected. It does not replace a later untouched time test.

## 1. Method choice and why

I use **logistic regression** as a simple classification model whose scores can rank pages. It is a reasonable first step for a binary decline outcome and lets me inspect coefficients. I use the same five earlier-window features from Week 3.

The pipeline fills missing values with training medians, scales features using training data, and fits logistic regression with `C=1`, `max_iter=1000`, and seed 42. I fix these settings before comparing the scores. I am not trying many models and reporting whichever looks best on this split.

The baseline stays `log(1 + impressions)`, doubled for CTR below 0.5% at positions 1–20. Its thresholds and multiplier are unchanged. A model must earn its extra complexity: my working goal remains at least five percentage points above the baseline on mean precision@20, followed by a robustness check.

I read the supplied [FlyRank paper](https://github.com/flyrank-bih/flyrank-ml-internship-starter/blob/main/docs/flyrank-seo-research-march-2026.pdf), especially the modelling appendix and methodology on pages 27, 29, and 36. It helped me separate descriptive model behaviour from a claim about future outcomes or editing benefit. I will review two specific findings in Week 6.

In [1]:
from pathlib import Path
import importlib.util
import subprocess
import sys

needed = {"duckdb": "duckdb", "huggingface_hub": "huggingface_hub", "sklearn": "scikit-learn",
          "pyarrow": "pyarrow", "matplotlib": "matplotlib", "pandas": "pandas"}
missing = [package for module, package in needed.items() if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "work/lane_utils.py").is_file()), None)
if root is None:
    if importlib.util.find_spec("google.colab") is None:
        raise RuntimeError("Open this notebook inside your repository checkout.")
    root = Path("/content/FlyrankAI-Internship")
    if not root.exists():
        subprocess.check_call(["git", "clone", "--depth", "1",
            "https://github.com/vibhanshu-s/FlyrankAI-Internship.git", str(root)])
sys.path.insert(0, str(root / "work"))
import lane_utils as u
import numpy as np
import pandas as pd
from IPython.display import display, Markdown

pages = u.load_pages(root)
experiment = u.labelled_pages(pages)
train_index, test_index = u.grouped_split(experiment)
training = experiment.iloc[train_index].copy()
test = experiment.iloc[test_index].copy()
print(f"March review pool: {len(pages):,} pages; known outcomes: {len(experiment):,}; unknown: {pages.future_decline.isna().sum():,}.")
print(f"Fixed client split: {len(training):,} training pages / {len(test):,} test pages.")
print("Features stop March 14; assumed decision March 18; labels use March 18–31. June remains sealed.")

March review pool: 60,534 pages; known outcomes: 57,459; unknown: 3,075.
Fixed client split: 14,070 training pages / 43,389 test pages.
Features stop March 14; assumed decision March 18; labels use March 18–31. June remains sealed.


## 2. Split design

I use the **same client-held-out split as Week 4**: 24 training clients and 8 test clients, with no client on both sides. The test clients contain more pages because client inventories are uneven. I report the page counts and the per-client metric rather than treating the split as 75%/25% of rows.

All inputs stop on March 14. The assumed decision is March 18, after a reporting gap, and the label uses March 18–31. Pages need complete measurements in both windows to enter evaluation. The 3,075 pages without later coverage still belong in a review pool, but their target cannot be evaluated here.

A grouped split checks transfer to different clients in this one month. It does not prove transfer to a different month or eliminate selection bias from the coverage filters. June stays sealed.

In [2]:
import json
baseline_receipt = json.loads((root / "work/outputs/w04_baseline_metrics.json").read_text(encoding="utf-8"))
assert baseline_receipt["contract"] == u.CONTRACT
assert baseline_receipt["baseline"] == u.BASELINE
assert baseline_receipt["data_fingerprint"] == u.fingerprint(pages)
assert baseline_receipt["split"] == u.split_fingerprint(experiment, train_index, test_index)
assert set(training.client_hash_id).isdisjoint(set(test.client_hash_id))
X_train, X_test = training[u.FEATURES], test[u.FEATURES]
y_train, y_test = training.future_decline.astype(int), test.future_decline.astype(int)
print("Matched the Week 4 data fingerprint, rule definition, and exact train/test page keys.")
print(f"Train decline rate: {100 * y_train.mean():.2f}%; test decline rate: {100 * y_test.mean():.2f}%.")
print("The target and later-window fields are separate from the five model inputs.")

Matched the Week 4 data fingerprint, rule definition, and exact train/test page keys.
Train decline rate: 31.68%; test decline rate: 38.25%.
The target and later-window fields are separate from the five model inputs.


## 3. Train and compare with my baseline

My primary metric is **mean precision@20 across test clients with full lists**. It measures the fraction of selected pages that later decline, averaged equally across those clients. Shorter lists are reported separately. The pool base rate uses the same full-list clients.

ROC-AUC and average precision give extra information about ordering across all test pages; larger clients can influence those pooled metrics. These numbers answer a decline-proxy question, not whether an editor should refresh a page.

In [3]:
model = u.make_model()
model.fit(X_train, y_train)
model_score = model.predict_proba(X_test)[:, 1]
baseline_score = u.rule_score(test)
results = {"Frozen Week 4 rule": u.evaluate(test, baseline_score),
           "Logistic regression": u.evaluate(test, model_score)}
assert np.isclose(results["Frozen Week 4 rule"]["macro_precision_at_20"],
                  baseline_receipt["metrics"]["macro_precision_at_20"])
comparison = u.summary_table(results)
display(comparison)
gain = results["Logistic regression"]["macro_precision_at_20"] - results["Frozen Week 4 rule"]["macro_precision_at_20"]
print(f"Model-minus-baseline mean precision@20: {100 * gain:+.2f} percentage points.")
print(f"Short-list clients: {results['Logistic regression']['short_queue_clients']}; "
      f"model decline share: {results['Logistic regression']['short_queue_mean']:.4f}.")
ranked = u.rank_frame(test, model_score)
ranked.to_parquet(root / "work/outputs/w05_predictions.parquet", index=False)
claim = ("The model meets my provisional five-point goal on this development split; it still needs a broader audit."
         if gain >= 0.05 else "The model does not meet my provisional five-point goal; I cannot justify switching from the rule yet.")
display(Markdown(claim))

,macro_precision_at_20,macro_base_rate_same_clients,roc_auc,average_precision,full_queue_clients,short_queue_clients,pages
Frozen Week 4 rule,0.3714,0.3981,0.5160,0.3811,7.0,1.0,43389.0
Logistic regression,0.5714,0.3981,0.6216,0.4876,7.0,1.0,43389.0


Model-minus-baseline mean precision@20: +20.00 percentage points.
Short-list clients: 1; model decline share: 0.3333.


The model meets my provisional five-point goal on this development split; it still needs a broader audit.

## 4. Errors and interpretation

I look at two operational error types: a selected page that does not meet the later decline target, and a declining page left outside its client's first 20. Neither tells me whether editing would have helped.

I use permutation importance to see how shuffling each feature changes ROC-AUC, and standardised coefficients to inspect direction inside the fitted model. Shared information between impressions, clicks, and CTR can make individual importance misleading. These are descriptions of model behaviour, not independent ranking factors or causal levers. The interpretation uses this development split and will not be used to tune a model on it.

In [4]:
from sklearn.inspection import permutation_importance
importance = permutation_importance(model, X_test, y_test, scoring="roc_auc", n_repeats=5, random_state=42)
interpretation = pd.DataFrame({"feature": u.FEATURES,
    "mean_auc_drop_when_shuffled": importance.importances_mean,
    "shuffle_std": importance.importances_std,
    "standardised_coefficient": model.named_steps["logisticregression"].coef_[0]})
interpretation = interpretation.sort_values("mean_auc_drop_when_shuffled", ascending=False)
display(interpretation.round(4))
print("Active-search-days coefficient is zero in this filtered slice; it contributes no learned variation here.")
print("The CTR coefficient is conditional on clicks and impressions; it is not a standalone causal effect.")
explanations = {
    "log_impressions": "earlier demand may distinguish stable inventory from more volatile pages, but client mix can explain it",
    "log_clicks": "clicks measure how much earlier visibility became visits; it shares information with CTR and demand",
    "ctr_pct": "click capture may mark pages worth investigating, but query intent and result-page layout are unmeasured",
    "weighted_position": "position summarises earlier search visibility, not the reason it will change",
    "active_search_days": "consistency of earlier visibility may matter, but it is only a short-window measure",
}
display(Markdown((chr(10) * 2).join(
    f"- **{feature}:** {explanations[feature]}." for feature in interpretation.feature.head(3))))

selected = ranked.client_rank.le(20)
print(f"Selected pages without later decline: {(selected & ranked.future_decline.eq(0)).sum():,}.")
print(f"Declining pages outside the first 20: {(~selected & ranked.future_decline.eq(1)).sum():,}.")
print("A 20-page budget leaves many declines unreviewed; this queue cannot cover the whole inventory.")
false_picks = ranked.loc[selected & ranked.future_decline.eq(0)].sort_values("score", ascending=False).head(3)
missed = ranked.loc[~selected & ranked.future_decline.eq(1)].sort_values("earlier_impressions", ascending=False).head(3)
errors = pd.concat([false_picks.assign(error="Selected, no observed decline"),
                    missed.assign(error="Declined, outside first 20")])
display(errors[u.KEYS + ["error", "client_rank", "score", "ctr_pct", "weighted_position",
                         "earlier_impressions", "later_impressions", "future_decline"]])
error_lines = []
for _, row in errors.iterrows():
    explanation = ("earlier signals looked risky, but the later window did not fall enough; query mix or normal variation could explain it"
        if row.future_decline == 0 else "earlier demand did not guarantee a high risk score; an unmeasured change or the limited review budget could explain the miss")
    error_lines.append(f"- **{row.content_hash_id}** — {row.error}. {explanation}. This is a possibility, not a verified cause.")
display(Markdown((chr(10) * 2).join(error_lines)))
assert len(errors) >= 3

import sklearn
receipt = {"contract": u.CONTRACT, "data_fingerprint": u.fingerprint(pages),
    "split": u.split_fingerprint(experiment, train_index, test_index), "baseline": u.BASELINE,
    "model": {"method": "LogisticRegression", "C": 1.0, "max_iter": 1000, "seed": 42,
              "imputation": "training median", "scaling": "training StandardScaler"},
    "metrics": results, "primary_gain_percentage_points": float(100 * gain),
    "provisional_five_point_goal_met": bool(gain >= 0.05),
    "feature_interpretation": interpretation.to_dict(orient="records"),
    "error_counts": {"selected_without_decline": int((selected & ranked.future_decline.eq(0)).sum()),
                     "declines_outside_first_20": int((~selected & ranked.future_decline.eq(1)).sum())},
    "versions": {"python": sys.version.split()[0], "scikit_learn": sklearn.__version__,
                 "pandas": pd.__version__, "numpy": np.__version__},
    "evidence_scope": "single-month, client-held-out development comparison; no causal editing claim"}
u.write_receipt(root, "w05_model_metrics.json", receipt)
print("Aggregate model receipt saved; individual predictions remain in an ignored parquet file.")

,feature,mean_auc_drop_when_shuffled,shuffle_std,standardised_coefficient
1,log_clicks,0.1150,0.0012,-0.6758
0,log_impressions,0.0791,0.0012,0.5001
2,ctr_pct,0.0040,0.0002,0.1489
3,weighted_position,0.0036,0.0002,0.0506
4,active_search_days,0.0000,0.0000,0.0000


Active-search-days coefficient is zero in this filtered slice; it contributes no learned variation here.
The CTR coefficient is conditional on clicks and impressions; it is not a standalone causal effect.


- **log_clicks:** clicks measure how much earlier visibility became visits; it shares information with CTR and demand.

- **log_impressions:** earlier demand may distinguish stable inventory from more volatile pages, but client mix can explain it.

- **ctr_pct:** click capture may mark pages worth investigating, but query intent and result-page layout are unmeasured.

Selected pages without later decline: 62.
Declining pages outside the first 20: 16,515.
A 20-page budget leaves many declines unreviewed; this queue cannot cover the whole inventory.


,client_hash_id,content_hash_id,error,client_rank,score,ctr_pct,weighted_position,earlier_impressions,later_impressions,future_decline
13732,client_23a62021009f63c4,content_b875a2f306635a58,"Selected, no observed decline",1,0.826468,0.000000,32.499097,30998.0,29878.0,0
10591,client_23a62021009f63c4,content_67a19b4e8f52924e,"Selected, no observed decline",6,0.797882,0.000000,44.433799,18119.0,15795.0,0
14724,client_23a62021009f63c4,content_d2def933ed902af2,"Selected, no observed decline",7,0.779805,0.000000,43.985897,14181.0,13959.0,0
15538,client_23a62021009f63c4,content_e8a52cf3d5988c07,"Declined, outside first 20",9591,0.171640,0.244969,16.161262,138793.0,86531.0,1
8770,client_23a62021009f63c4,content_36e53e9c707674fc,"Declined, outside first 20",7113,0.281102,0.104614,33.144111,105148.0,73675.0,1
24361,client_62f4a7e64f5e0096,content_7c6373141eae744a,"Declined, outside first 20",4381,0.352031,0.055846,5.990078,84160.0,41458.0,1


- **content_b875a2f306635a58** — Selected, no observed decline. earlier signals looked risky, but the later window did not fall enough; query mix or normal variation could explain it. This is a possibility, not a verified cause.

- **content_67a19b4e8f52924e** — Selected, no observed decline. earlier signals looked risky, but the later window did not fall enough; query mix or normal variation could explain it. This is a possibility, not a verified cause.

- **content_d2def933ed902af2** — Selected, no observed decline. earlier signals looked risky, but the later window did not fall enough; query mix or normal variation could explain it. This is a possibility, not a verified cause.

- **content_e8a52cf3d5988c07** — Declined, outside first 20. earlier demand did not guarantee a high risk score; an unmeasured change or the limited review budget could explain the miss. This is a possibility, not a verified cause.

- **content_36e53e9c707674fc** — Declined, outside first 20. earlier demand did not guarantee a high risk score; an unmeasured change or the limited review budget could explain the miss. This is a possibility, not a verified cause.

- **content_7c6373141eae744a** — Declined, outside first 20. earlier demand did not guarantee a high risk score; an unmeasured change or the limited review budget could explain the miss. This is a possibility, not a verified cause.

Aggregate model receipt saved; individual predictions remain in an ignored parquet file.


## 5. Self-check

- [x] I explained why logistic regression fits the ranking question and kept its settings fixed.
- [x] I used the same page keys, label, split, baseline rule, and metric as Week 4.
- [x] Imputation and scaling were fitted only on training pages.
- [x] I reported the base rate, shorter queues, feature interpretation, and actual error cases.
- [x] I separated decline risk from the value of refreshing content.
- [x] The notebook ran top to bottom; its aggregate metrics receipt is committed with it.
- [x] Data, prediction files, and credentials are excluded from git.

The final comparison below is the table I will carry into the Week 6 validation audit.

In [5]:
display(comparison)
print(claim)
print("Same test pages, same metric, same frozen baseline; development evidence only.")

,macro_precision_at_20,macro_base_rate_same_clients,roc_auc,average_precision,full_queue_clients,short_queue_clients,pages
Frozen Week 4 rule,0.3714,0.3981,0.5160,0.3811,7.0,1.0,43389.0
Logistic regression,0.5714,0.3981,0.6216,0.4876,7.0,1.0,43389.0


The model meets my provisional five-point goal on this development split; it still needs a broader audit.
Same test pages, same metric, same frozen baseline; development evidence only.
